# Chatterbox full Arabic voice test

Full Arabic alphabet, numbers, and paragraph tests. TTS input is **only** the target utterance.

## 1. Chatterbox environment bootstrap

In [ ]:
import os, subprocess, sys

def _chatterbox_import_ok():
    try:
        import chatterbox  # noqa: F401
        from transformers import __version__ as tv
        return tv.startswith("5.")
    except Exception:
        return False

if os.environ.get("CB_NB_SKIP_BOOTSTRAP") == "1" and _chatterbox_import_ok():
    print("[bootstrap] skipped (CB_NB_SKIP_BOOTSTRAP=1, chatterbox OK)")
elif os.environ.get("CB_NB_SKIP_BOOTSTRAP") == "1":
    print("[bootstrap] fixing deps for Chatterbox (transformers 5.x / numpy<2)...")
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "perth"], check=False)
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "setuptools<81", "resemble-perth", "chatterbox-tts", "huggingface_hub",
        "soundfile", "torch", "torchaudio",
    ])
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--force-reinstall",
        "transformers==5.2.0", "safetensors==0.5.3", "numpy>=1.24,<2.0",
    ])
    print("Chatterbox deps repaired.")
else:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "perth"], check=False)
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "setuptools<81", "resemble-perth", "chatterbox-tts", "huggingface_hub",
        "soundfile", "torch", "torchaudio", "ipython", "jupyter", "nbconvert", "notebook",
    ])
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--force-reinstall",
        "transformers==5.2.0", "safetensors==0.5.3", "numpy>=1.24,<2.0",
    ])
    print("Chatterbox bootstrap done. Set CB_NB_SKIP_BOOTSTRAP=1 for faster re-runs.")


[bootstrap] skipped (CB_NB_SKIP_BOOTSTRAP=1, chatterbox OK)


## 2. Paths and model files

In [2]:
import gc
import time
import traceback
from pathlib import Path

BASE_DIR = Path.cwd().resolve()
CHATTERBOX_DIR = BASE_DIR / "laghtna-chatterbox-v1 model"
AUDIO_ROOT = BASE_DIR / "generated_audio"
DIR_ALPHABET = AUDIO_ROOT / "chatterbox_alphabet"
DIR_NUMBERS = AUDIO_ROOT / "chatterbox_numbers"
DIR_PARAGRAPHS = AUDIO_ROOT / "chatterbox_paragraphs"
CACHE_DIR = AUDIO_ROOT / "_cache"

for d in (DIR_ALPHABET, DIR_NUMBERS, DIR_PARAGRAPHS, CACHE_DIR):
    d.mkdir(parents=True, exist_ok=True)

EXPECTED_CB = [
    "conds.pt", "mtl_tokenizer.json", "s3gen.safetensors", "t3_cfg.safetensors",
    "t3_mtl23ls_v2.safetensors", "tokenizer.json", "ve.safetensors",
]
print("cwd:", BASE_DIR)
print("\nlaghtna-chatterbox-v1 model files:")
for name in EXPECTED_CB:
    p = CHATTERBOX_DIR / name
    print(f"  [{'OK' if p.is_file() else 'MISS'}] {name}")


cwd: C:\Users\sondo\Desktop\Voice Model Stuff

laghtna-chatterbox-v1 model files:
  [OK] conds.pt
  [OK] mtl_tokenizer.json
  [OK] s3gen.safetensors
  [OK] t3_cfg.safetensors
  [OK] t3_mtl23ls_v2.safetensors
  [OK] tokenizer.json
  [OK] ve.safetensors


## 3. Device and dependencies

In [3]:
import inspect

HAVE_CHATTERBOX = False
HAVE_SAFE = False
HAVE_HF = False
HAVE_SF = False
torch = None
CUDA_AVAILABLE = False
DEVICE_PREF = "cpu"
GPU_NAME = None

try:
    import torch
    CUDA_AVAILABLE = torch.cuda.is_available()
    DEVICE_PREF = "cuda" if CUDA_AVAILABLE else "cpu"
    if CUDA_AVAILABLE:
        GPU_NAME = torch.cuda.get_device_name(0)
except ImportError:
    print("torch missing")

print("CUDA available:", CUDA_AVAILABLE)
print("GPU name:", GPU_NAME or "(n/a)")
print("Selected device:", DEVICE_PREF)

for mod in ("safetensors", "huggingface_hub", "soundfile"):
    try:
        __import__(mod)
        print("[deps OK]", mod)
        if mod == "safetensors":
            HAVE_SAFE = True
        if mod == "huggingface_hub":
            HAVE_HF = True
        if mod == "soundfile":
            HAVE_SF = True
    except ImportError as e:
        print("[deps MISSING]", mod, e)

try:
    import chatterbox  # noqa: F401
    HAVE_CHATTERBOX = True
    print("[deps OK] chatterbox")
except ImportError:
    print("[deps MISSING] chatterbox-tts")
    traceback.print_exc()


CUDA available: False
GPU name: (n/a)
Selected device: cpu
[deps OK] safetensors
[deps OK] huggingface_hub
[deps OK] soundfile
[deps OK] chatterbox


## 4. Egyptian mappings and paragraphs

In [4]:
LETTER_PRON = {
    "ا": "أَلِف", "أ": "أَلِف", "ب": "بِه", "ت": "تِه", "ث": "سِه",
    "ج": "جِيم", "ح": "حَا", "خ": "خَا", "د": "دَال", "ذ": "ذَال",
    "ر": "رَا", "ز": "زَا", "س": "سِين", "ش": "شِين", "ص": "صَاد",
    "ض": "ضَاد", "ط": "طَا", "ظ": "ظَا", "ع": "عِين", "غ": "غِين",
    "ف": "فَا", "ق": "قَاف", "ك": "كَاف", "ل": "لَام", "م": "مِيم",
    "ن": "نُون", "ه": "هَا", "و": "وَاو", "ي": "يَا",
}
LETTERS_ORDER = list("اأبتثجحخدذرزسشصضطظعغفقكلمنهوي")
NUMBER_PRON = {
    0: "صِفْر", 1: "وَاحِد", 2: "اِتْنِين", 3: "تَلَاتَه", 4: "أَرْبَعَه",
    5: "خَمْسَه", 6: "سِتَّه", 7: "سَبْعَه", 8: "تَمَانْيَه", 9: "تِسْعَه", 10: "عَشَرَه",
}

def letter_tts(letter: str) -> str:
    base = LETTER_PRON.get(letter, letter)
    return base if base.endswith(".") else base + "."

def number_tts(n: int) -> str:
    base = NUMBER_PRON[n]
    return base if base.endswith(".") else base + "."

PARAGRAPHS = [
"""هلاً بك! وجدت لك شقتين ممتازتين في التجمع.

الأولى شقة في شمال الرحاب بالتجمع الأول، مساحتها 192 متر مربع، بتلات غرف نوم وحمامين، وسعرها 5,768,000 جنيه، وميزة ليها باركينج مغطى.

فيه شقة تانية في النرجس الجديدة بالتجمع الخامس، مساحتها 150 متر مربع، بتلات غرف نوم وحمامين، وسعرها 3,750,000 جنيه، وميزتها قربها من شارع التسعين.

أنصحك بمعاينة الخيارين لتقييم الأنسب ليك. هل تحب تعرف تفاصيل أكتر عن أي شقة منهم؟""",
"""هلاً بك! لقيت لك مكتب إداري كويس أوي في التجمع، تحديداً في منطقة الياسمين 2 بالقاهرة الجديدة. المكتب ده تشطيب كامل بالتكييف ومساحته 62 متر مربع، وسعره 7,130,000 جنيه مصري. كمان فيه أنظمة سداد لحد 7 سنين. لو حابب تشوف حاجة بمساحة مختلفة أو تفاصيل أكتر ممكن تقولي.""",
"""تتطلع للحصول على شقة أو فيلا في التجمع؟

أفضل مطابقة: شقة مميزة جداً للبيع في شمال الرحاب، التجمع
الموقع: شمال الرحاب، القاهرة الجديدة، القاهرة
السعر: 5,768,000 جم
مساحة: 192.0 م²
غرف نوم/حمامات: 3/2
مميزات: موقف سيارات مغطي، خط ساخن، أمان

خيار ثانٍ: شقفة للبيع في نجارة الجديدة
الموقع: نجارة الجديدة، القاهرة الجديدة، القاهرة
السعر: 3,750,000 جم
مساحة: 150.0 م²
غرف نوم/حمامات: 3/2
مميزات: كهرباء مستقلة، غاز طبيعي

ننصح بالاختيار الأول لقربه من التجمع مباشرة ومرافقه الأساسية. هل ترغب في المزيد من المعلومات حول الشقة الأولى؟""",
]


## 5. Load Chatterbox

In [5]:
from IPython.display import Audio as IPA, display

GENERATED = {"alphabet": [], "numbers": [], "paragraphs": []}
CHATTERBOX_API = None
CHATTER_DEVICE = ""
CHATTER_DIALECT_MSG = ""
CHATTER_RP_USED = ""
MODEL_LOADED = False
S3GEN_SR = 24000

def format_bytes(nb: int) -> str:
    if nb < 1024:
        return f"{nb} B"
    if nb < 1024**2:
        return f"{nb / 1024:.2f} KB"
    return f"{nb / 1024 / 1024:.2f} MB"

def unload_cuda(verbose=False):
    gc.collect()
    if torch is not None and CUDA_AVAILABLE:
        torch.cuda.empty_cache()
    if verbose:
        print("[gc+cuda-empty_cache]")

def load_chatterbox():
    global CHATTERBOX_API, CHATTER_DEVICE, MODEL_LOADED, CHATTER_DIALECT_MSG, S3GEN_SR
    if not (HAVE_CHATTERBOX and HAVE_SAFE and HAVE_HF and torch is not None):
        print("Missing deps for Chatterbox")
        return None

    try:
        import sys as _sy
        import perth.perth_net as _pnp
        _pm = _sy.modules.get("perth")
        if _pm is not None and not callable(getattr(_pm, "PerthImplicitWatermarker", None)):
            setattr(_pm, "PerthImplicitWatermarker", _pnp.PerthImplicitWatermarker)
    except Exception as pw:
        print("Perth bridge:", pw)

    from huggingface_hub import hf_hub_download
    from safetensors.torch import load_file
    from chatterbox.mtl_tts import ChatterboxMultilingualTTS, Conditionals, SUPPORTED_LANGUAGES
    from chatterbox.models.t3 import T3
    from chatterbox.models.t3.modules.t3_config import T3Config
    from chatterbox.models.s3gen import S3Gen, S3GEN_SR as _SR
    from chatterbox.models.voice_encoder import VoiceEncoder
    from chatterbox.models.tokenizers.tokenizer import MTLTokenizer

    S3GEN_SR = int(_SR)
    grapheme_path = hf_hub_download(
        repo_id="ResembleAI/chatterbox",
        filename="grapheme_mtl_merged_expanded_v1.json",
        cache_dir=str(CACHE_DIR),
    )

    order = ["cuda", "cpu"] if DEVICE_PREF == "cuda" and CUDA_AVAILABLE else ["cpu"]
    api_obj = None
    for dv in order:
        t0 = time.perf_counter()
        try:
            ck = CHATTERBOX_DIR
            ve = VoiceEncoder()
            ve.load_state_dict(load_file(str(ck / "ve.safetensors")), strict=False)
            ve.eval()
            sg = S3Gen()
            sg.load_state_dict(load_file(str(ck / "s3gen.safetensors")), strict=False)
            t3 = T3(T3Config.multilingual())
            t3.load_state_dict(load_file(str(ck / "t3_mtl23ls_v2.safetensors")), strict=False)
            ve, sg, t3 = ve.to(dv).eval(), sg.to(dv).eval(), t3.to(dv).eval()
            tokzr = MTLTokenizer(grapheme_path)
            cond_path = ck / "conds.pt"
            conds = (
                Conditionals.load(str(cond_path), map_location=torch.device("cpu")).to(dv)
                if cond_path.is_file()
                else None
            )
            api_obj = ChatterboxMultilingualTTS(t3, sg, ve, tokzr, dv, conds=conds)
            CHATTER_DEVICE = dv
            MODEL_LOADED = True
            print(f"Model loaded on {dv} in {time.perf_counter() - t0:.2f}s")
            break
        except torch.cuda.OutOfMemoryError:
            unload_cuda(False)
            api_obj = None
        except Exception:
            traceback.print_exc()
            api_obj = None

    if api_obj is None:
        MODEL_LOADED = False
        return None

    langs = SUPPORTED_LANGUAGES
    dialect_hit = sorted({"ar-eg", "arq", "ar_eg", "eg", "egy", "masri"} & {str(k).lower() for k in langs})
    if dialect_hit:
        CHATTER_DIALECT_MSG = dialect_hit[0]
        print("Chatterbox selected dialect/voice:", CHATTER_DIALECT_MSG)
    else:
        CHATTER_DIALECT_MSG = "default (Laghtna Egyptian weights in conds.pt)"
        print("No explicit ar-EG in SUPPORTED_LANGUAGES; using Laghtna local conds (Egyptian Arabic).")

    arabic_id = next((c for c in ("ar", "arb") if c in langs), "ar")
    print("Chatterbox language_id used:", arabic_id, f"({langs.get(arabic_id, '')})")

    CHATTERBOX_API = api_obj
    return api_obj

def _base_gen_kw(api_obj):
    kw = dict(language_id="ar", audio_prompt_path=None, exaggeration=0.5, cfg_weight=0.45, temperature=0.75)
    sig = inspect.signature(api_obj.generate)
    return kw, sig

def chatterbox_generate(text: str, rp_values=(1.2, 1.3, 1.5)):
    global CHATTER_RP_USED
    api_obj = CHATTERBOX_API
    if api_obj is None:
        return None, float("nan"), ""
    base_kw, sig = _base_gen_kw(api_obj)
    last_err = None
    for rp in rp_values:
        gen_kw = dict(base_kw)
        if "repetition_penalty" in sig.parameters:
            gen_kw["repetition_penalty"] = rp
            CHATTER_RP_USED = str(rp)
        else:
            CHATTER_RP_USED = "unsupported"
        try:
            with torch.inference_mode():
                t0 = time.perf_counter()
                wav = api_obj.generate(text[:5000], **gen_kw)
            return wav, time.perf_counter() - t0, CHATTER_RP_USED
        except Exception as e:
            last_err = e
            if "repetition" in str(e).lower():
                continue
    if last_err:
        print("generate failed:", last_err)
    return None, float("nan"), CHATTER_RP_USED

def save_chatter_wav(wav_tensor, out_path: Path):
    sr = int(getattr(CHATTERBOX_API, "sr", S3GEN_SR))
    wav_np = wav_tensor.squeeze(0).detach().cpu().numpy().astype("float32")
    if HAVE_SF:
        import soundfile as sf
        sf.write(str(out_path), wav_np, sr, subtype="PCM_16")
    else:
        import torchaudio as ta
        w = wav_tensor.squeeze(0).detach().cpu()
        ta.save(str(out_path), w.unsqueeze(0) if w.dim() == 1 else w, sr)
    return out_path.is_file()

load_chatterbox()
if CHATTER_RP_USED == "" and CHATTERBOX_API is not None:
    _, sig0 = _base_gen_kw(CHATTERBOX_API)
    if "repetition_penalty" in sig0.parameters:
        CHATTER_RP_USED = "1.2 (default)"
        print("Chatterbox repetition_penalty: will use 1.2, then 1.3/1.5 on repetition issues")
    else:
        CHATTER_RP_USED = "unsupported"
        print("Chatterbox repetition_penalty: unsupported by this build")


C:\Users\sondo\AppData\Local\Programs\Python\Python311\Lib\site-packages\perth\perth_net\__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


C:\Users\sondo\AppData\Local\Programs\Python\Python311\Lib\site-packages\diffusers\models\lora.py:393: FutureWarning: `LoRACompatibleLinear` is deprecated and will be removed in version 1.0.0. Use of `LoRACompatibleLinear` is deprecated. Please switch to PEFT backend by installing PEFT: `pip install peft`.
  deprecate("LoRACompatibleLinear", "1.0.0", deprecation_message)


loaded PerthNet (Implicit) at step 250,000
Model loaded on cpu in 16.54s
No explicit ar-EG in SUPPORTED_LANGUAGES; using Laghtna local conds (Egyptian Arabic).
Chatterbox language_id used: ar (Arabic)
Chatterbox repetition_penalty: will use 1.2, then 1.3/1.5 on repetition issues


## 6. Alphabet tests

In [6]:
MAX_PLAYBACK = 5
seen_letters = set()
ok_alpha = 0

for letter in LETTERS_ORDER:
    if letter in seen_letters:
        continue
    seen_letters.add(letter)
    tts_text = letter_tts(letter)
    slug = f"letter_u{ord(letter):04x}"
    out_path = DIR_ALPHABET / f"{slug}.wav"
    print("Original letter:", letter)
    print("Final text sent to TTS:", tts_text)
    wav, infer_s, rp = chatterbox_generate(tts_text)
    if wav is not None and save_chatter_wav(wav, out_path):
        ok_alpha += 1
        GENERATED["alphabet"].append(str(out_path.resolve()))
        print("Saved:", out_path.resolve(), "| inference:", f"{infer_s:.3f}s", "| rp:", rp)
        if ok_alpha <= MAX_PLAYBACK:
            display(IPA(filename=str(out_path)))
    else:
        print("FAILED:", out_path)
    print("---")

print(f"Alphabet: {ok_alpha}/{len(seen_letters)} wav files")
if ok_alpha > MAX_PLAYBACK:
    for p in GENERATED["alphabet"][MAX_PLAYBACK:]:
        print(" ", p)


C:\Users\sondo\AppData\Local\Programs\Python\Python311\Lib\contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
The following generation flags are not valid and may be ignored: ['output_attentions']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Original letter: ا
Final text sent to TTS: أَلِف.



Sampling:   0%|          | 0/1000 [00:00<?, ?it/s]


Sampling:   0%|          | 1/1000 [00:00<02:57,  5.61it/s]


Sampling:   0%|          | 2/1000 [00:00<02:37,  6.34it/s]

...


Sampling:   0%|          | 3/1000 [00:00<02:22,  6.97it/s]


Sampling:   0%|          | 4/1000 [00:00<02:22,  6.99it/s]


Sampling:   0%|          | 5/1000 [00:00<02:49,  5.86it/s]


Sampling:   0%|          | 5/1000 [00:00<02:41,  6.18it/s]

Saved: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\chatterbox_alphabet\letter_u064a.wav | inference: 9.304s | rp: 1.2
---
Alphabet: 29/29 wav files
  C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\chatterbox_alphabet\letter_u062c.wav
  C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\chatterbox_alphabet\letter_u062d.wav
  C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\chatterbox_alphabet\letter_u062e.wav
  C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\chatterbox_alphabet\letter_u062f.wav
  C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\chatterbox_alphabet\letter_u0630.wav
  C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\chatterbox_alphabet\letter_u0631.wav
  C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\chatterbox_alphabet\letter_u0632.wav
  C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\chatterbox_alphabet\letter_u0633.wav
  C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\chatterbox_a

## 7. Number tests

In [7]:
ok_num = 0
for n in range(11):
    tts_text = number_tts(n)
    out_path = DIR_NUMBERS / f"num_{n:02d}.wav"
    print("Number:", n)
    print("Final text sent to TTS:", tts_text)
    wav, infer_s, rp = chatterbox_generate(tts_text)
    if wav is not None and save_chatter_wav(wav, out_path):
        ok_num += 1
        GENERATED["numbers"].append(str(out_path.resolve()))
        print("Saved:", out_path.resolve(), "| inference:", f"{infer_s:.3f}s", "| rp:", rp)
        if ok_num <= MAX_PLAYBACK:
            display(IPA(filename=str(out_path)))
    else:
        print("FAILED:", out_path)
    print("---")
print(f"Numbers: {ok_num}/11 wav files")


Number: 0
Final text sent to TTS: صِفْر.



Sampling:   0%|          | 0/1000 [00:00<?, ?it/s]


Sampling:   0%|          | 1/1000 [00:00<02:47,  5.96it/s]


Sampling:   0%|          | 2/1000 [00:00<02:40,  6.20it/s]


Sampling:   0%|          | 3/1000 [00:00<02:33,  6.51it/s]

...


Sampling:   3%|▎         | 26/1000 [00:03<02:14,  7.23it/s]


Sampling:   3%|▎         | 27/1000 [00:03<02:13,  7.28it/s]


Sampling:   3%|▎         | 28/1000 [00:03<02:11,  7.41it/s]


Sampling:   3%|▎         | 28/1000 [00:03<02:13,  7.29it/s]

Saved: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\chatterbox_numbers\num_10.wav | inference: 13.374s | rp: 1.2
---
Numbers: 11/11 wav files


## 8. Paragraph tests (one at a time)

In [8]:
ok_para = 0
for i, para in enumerate(PARAGRAPHS, start=1):
    tts_text = para.strip()
    out_path = DIR_PARAGRAPHS / f"paragraph_{i}.wav"
    print(f"\n=== Paragraph {i} ===")
    print("Final text sent to TTS:")
    print(tts_text[:200], "..." if len(tts_text) > 200 else "")
    wav, infer_s, rp = chatterbox_generate(tts_text)
    if wav is not None and save_chatter_wav(wav, out_path):
        ok_para += 1
        GENERATED["paragraphs"].append(str(out_path.resolve()))
        print("Saved:", out_path.resolve())
        print("Inference time (s):", f"{infer_s:.3f}", "| repetition_penalty:", rp)
        display(IPA(filename=str(out_path)))
    else:
        print("FAILED:", out_path)
    unload_cuda(verbose=True)

print(f"Paragraphs: {ok_para}/{len(PARAGRAPHS)} wav files")



=== Paragraph 1 ===
Final text sent to TTS:
هلاً بك! وجدت لك شقتين ممتازتين في التجمع.

الأولى شقة في شمال الرحاب بالتجمع الأول، مساحتها 192 متر مربع، بتلات غرف نوم وحمامين، وسعرها 5,768,000 جنيه، وميزة ليها باركينج مغطى.

فيه شقة تانية في النر ...



Sampling:   0%|          | 0/1000 [00:00<?, ?it/s]


Sampling:   0%|          | 1/1000 [00:00<03:03,  5.45it/s]


Sampling:   0%|          | 2/1000 [00:00<02:50,  5.87it/s]


Sampling:   0%|          | 3/1000 [00:00<02:42,  6.12it/s]

...


Sampling:   9%|▉         | 92/1000 [00:15<02:27,  6.15it/s]


Sampling:   9%|▉         | 93/1000 [00:15<02:27,  6.16it/s]


Sampling:   9%|▉         | 93/1000 [00:15<02:29,  6.05it/s]

Saved: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\chatterbox_paragraphs\paragraph_3.wav
Inference time (s): 31.931 | repetition_penalty: 1.2


[gc+cuda-empty_cache]
Paragraphs: 3/3 wav files


## 9. Summary

In [9]:
all_paths = GENERATED["alphabet"] + GENERATED["numbers"] + GENERATED["paragraphs"]
print("\nDevice used:", CHATTER_DEVICE)
print("Dialect/voice:", CHATTER_DIALECT_MSG)
print("repetition_penalty:", CHATTER_RP_USED)
print("\n=== Generated files summary ===")
for p in all_paths:
    print(p)
print(f"\nTotal: {len(all_paths)} files")

if MODEL_LOADED:
    print("✅ Model loaded")
else:
    print("❌ Model loaded")
if len(GENERATED["alphabet"]) >= len(set(LETTERS_ORDER)):
    print("✅ Alphabet tests generated")
else:
    print("❌ Alphabet tests generated")
if len(GENERATED["numbers"]) >= 11:
    print("✅ Number tests generated")
else:
    print("❌ Number tests generated")
if len(GENERATED["paragraphs"]) >= len(PARAGRAPHS):
    print("✅ Paragraph tests generated")
else:
    print("❌ Paragraph tests generated")
if all_paths and all(Path(p).is_file() for p in all_paths):
    print("✅ Audio files saved")
else:
    print("❌ Audio files saved")
if MODEL_LOADED and all_paths:
    print("✅ Playback widgets displayed")
else:
    print("❌ Playback widgets displayed")



Device used: cpu
Dialect/voice: default (Laghtna Egyptian weights in conds.pt)
repetition_penalty: 1.2

=== Generated files summary ===
C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\chatterbox_alphabet\letter_u0627.wav
C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\chatterbox_alphabet\letter_u0623.wav
C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\chatterbox_alphabet\letter_u0628.wav
C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\chatterbox_alphabet\letter_u062a.wav
C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\chatterbox_alphabet\letter_u062b.wav
C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\chatterbox_alphabet\letter_u062c.wav
C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\chatterbox_alphabet\letter_u062d.wav
C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\chatterbox_alphabet\letter_u062e.wav
C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\chatterbox_alphabet\letter_u062f.wav
C:\Users\sondo\De